In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("smartcart_customers.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

# data preprocessing

## handle missing values

In [ ]:
df["Income"]=df["Income"].fillna(df["Income"].median())

In [ ]:
df.isnull().sum()

## feature engineering


In [ ]:
# age
df["Age"]=2026-df["Year_Birth"]

In [ ]:
df.head()

In [ ]:
#Customer joinging date
df["Dt_Customer"]=pd.to_datetime(df["Dt_Customer"],dayfirst=True)

reference_date=df["Dt_Customer"].max()

df["Customer_tenure_days"]=(reference_date-df["Dt_Customer"]).dt.days

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
# spending

df["Total_spending"]=df["MntWines"] + df["MntFruits"]+df["MntMeatProducts"]+df["MntFishProducts"]+df["MntSweetProducts"]

In [ ]:
df.head()

In [ ]:
#children

df["Total_children"]=df["Kidhome"]+df["Teenhome"]

In [ ]:
df.head()

In [ ]:
#education

df["Education"].value_counts()

df["Education"]=df["Education"].replace({
    "Basic":"Undergraduate", "2n Cycle":"Undergraduate",
    "Graduation":"Graduate",
    "Master":"Postgraduate","PhD":"Postgraduate"
})



In [ ]:
df["Education"].value_counts()

In [ ]:
df.head()

In [ ]:
#marital status
df["Living_with"]=df["Marital_Status"].replace({
    "Married":"Partner", "Together":"Partner",
    "Single":"Alone","Divorced":"Alone","Widow":"Alone",
    "Absurd":"Alone","YOLO":"Alone"
})


In [ ]:

df["Living_with"].value_counts()

## drop columns

In [ ]:
#remove customer id
df.drop("ID",axis=1)

In [ ]:
#removing other unnecessary columns
cols=["ID","Year_Birth","Marital_Status","Kidhome","Teenhome","Dt_Customer"]
spending_cols=["MntWines","MntFruits","MntMeatProducts","MntFishProducts","MntSweetProducts","MntGoldProds"]

cols_to_drop=cols+spending_cols

df_cleaned=df.drop(columns=cols_to_drop)

In [ ]:
df_cleaned

In [ ]:
df_cleaned.head()

# outliers

In [ ]:
cols=["Income","Recency","Response","Age","Total_spending","Total_children"]

#relative plots of some features -pair plots

sns.pairplot(df_cleaned[cols])

In [ ]:
#remove outliers

print("data size with outliers:",len(df_cleaned))

df_cleaned=df_cleaned[(df_cleaned["Age"]<90)]
df_cleaned=df_cleaned[(df_cleaned["Income"]<600_000)]

print("data size without outliers:",len(df_cleaned))


# heatmap

In [ ]:
corr=df_cleaned.corr(numeric_only=True)

In [ ]:
corr

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(
    corr,
    annot=True,
    annot_kws={"size":6},
    cmap="coolwarm"
)

In [ ]:
df_cleaned.head()

# encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
ohe=OneHotEncoder()

cat_cols=["Education","Living_with"]

enco_cols=ohe.fit_transform(df_cleaned[cat_cols])

In [ ]:
enco_df=pd.DataFrame(enco_cols.toarray(),columns=ohe.get_feature_names_out(cat_cols),index=df_cleaned.index)



In [ ]:
enco_df.head()

In [ ]:
df_encoded=pd.concat([df_cleaned.drop(columns=cat_cols),enco_df],axis=1)

In [ ]:
df_encoded.shape

In [ ]:
df_encoded.head()

# scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X=df_encoded

In [ ]:
scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

In [ ]:
X_scaled

# visualize



In [ ]:
X_scaled.shape

In [ ]:
#2D 
from sklearn.decomposition import PCA

In [ ]:
pca=PCA(n_components=2)

X_pca=pca.fit_transform(X_scaled)

In [ ]:
#plot

plt.scatter(X_pca[:,0],X_pca[:,1])

In [ ]:
pca.explained_variance_ratio_

In [ ]:
#3D

pca=PCA(n_components=3)

X_pca=pca.fit_transform(X_scaled)

pca.explained_variance_ratio_

In [ ]:
#plot
fig=plt.figure(figsize=(8,6))

ax=fig.add_subplot(111,projection="3d")
ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2])

ax.set_xlabel("PCA1")
ax.set_ylabel("PCA2")
ax.set_zlabel("PCA3")
ax.set_title("3D projection")

# Analyze K value
## 1.Elbow method

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"

from sklearn.cluster import KMeans
from kneed import KneeLocator

wcss=[]
for k in range(1,11):
    kmeans=KMeans(n_clusters=k,random_state=42)
    kmeans.fit_predict(X_pca)
    wcss.append(kmeans.inertia_)

In [ ]:
knee=KneeLocator(range(1,11),wcss,curve="convex",direction="decreasing")
optimal_k=knee.elbow

In [ ]:
optimal_k

In [ ]:
print("best k is ",optimal_k)

In [ ]:
#plot
plt.plot(range(1,11),wcss,marker='o')
plt.xlabel("K")
plt.ylabel("WCSS")

## 2.silhouette score

In [ ]:
from sklearn.metrics import silhouette_score

scores=[]

for k in range(2,11):
    kmeans=KMeans(n_clusters=k,random_state=42)
    labels=kmeans.fit_predict(X_pca)

    score=silhouette_score(X_pca,labels)
    scores.append(score)

##plot
plt.plot(range(2,11),scores,marker='o')
plt.xlabel("K")
plt.ylabel("silhouette score")


In [ ]:
#combined plot

k_range=range(2,11)

fig,ax1=plt.subplots(figsize=(8,6))

ax1.plot(k_range,wcss[:len(k_range)],marker='o',color="blue")
ax1.set_xlabel("K")
ax1.set_ylabel("WCSS")

ax2=ax1.twinx()
ax2.plot(k_range,scores[:len(k_range)],marker="x",color="red",linestyle="--")
ax2.set_ylabel("SS")

# clustering

In [ ]:
#k_means

kmeans=KMeans(n_clusters=4,random_state=42)
labels=kmeans.fit_predict(X_pca)

In [ ]:
fig=plt.figure(figsize=(8,6))

ax=fig.add_subplot(111,projection="3d")
ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2],c=labels)

In [ ]:
#agglomerative clustering
from sklearn.cluster import AgglomerativeClustering


In [ ]:
agg_clf=AgglomerativeClustering(n_clusters=4,linkage="ward")
labels_agg=agg_clf.fit_predict(X_pca)

In [ ]:
fig=plt.figure(figsize=(8,6))

ax=fig.add_subplot(111,projection="3d")
ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2],c=labels_agg)

# characterization of clusters


In [ ]:
X["clusters"]=labels_agg


In [ ]:
X.head()

In [ ]:
pal=["red","blue","yellow","green"]

sns.countplot(x=X["clusters"],palette=pal,hue=X["clusters"])

In [ ]:
# income and spending patterns

sns.scatterplot(x=X["Total_spending"],y=X["Income"],hue=X["clusters"],palette=pal)

In [ ]:
# cluster summary
cluster_summary=X.groupby("clusters").mean()
print(cluster_summary)